In [1]:
# Manipulation des tables Parquet et Arrow.
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs

# Fonction scikit-learn utilisée pour diviser les données.
from sklearn.model_selection import train_test_split

In [2]:
# Crée l'objet PyArrow qui communique avec Amazon S3.
s3 = fs.S3FileSystem(region="us-east-1")

# Nom du bucket S3.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Fichier source contenant les données nettoyées et les ID.
SOURCE_PATH = (
    f"{BUCKET_NAME}/raw/parquet/"
    "mushrooms_CleanedNamesID.parquet"
)

# Dossier de destination pour les trois datasets.
OUTPUT_DIR = f"{BUCKET_NAME}/processed/parquet"

# Chemins complets des fichiers de sortie.
TRAIN_PATH = f"{OUTPUT_DIR}/train.parquet"
VALIDATION_PATH = f"{OUTPUT_DIR}/validation.parquet"
TEST_PATH = f"{OUTPUT_DIR}/test.parquet"

In [3]:
# Lit le fichier Parquet directement depuis S3.
table = pq.read_table(
    SOURCE_PATH,
    filesystem=s3
)

# Affiche quelques informations pour vérifier le chargement.
print("Nombre de lignes :", table.num_rows)
print("Colonnes :", table.column_names)

In [4]:
# Convertit temporairement la table PyArrow en DataFrame Pandas.
dataframe = table.to_pandas()

# Affiche les cinq premières lignes.
display(dataframe.head())

In [5]:
# Sépare 70 % des données pour le train.
#
# Les 30 % restants seront ensuite partagés entre
# validation et test.
train_df, remaining_df = train_test_split(
    dataframe,
    test_size=0.30,
    random_state=42,
    stratify=dataframe["mushroom_name"]
)

In [6]:
# Divise les 30 % restants en deux parties égales.
#
# Cela donne :
# - 15 % validation
# - 15 % test
validation_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=42,
    stratify=remaining_df["mushroom_name"]
)

# Calcule la taille totale du dataset.
total_rows = len(dataframe)

# Affiche les proportions réelles obtenues.
print(f"Train : {len(train_df) / total_rows:.2%}")
print(f"Validation : {len(validation_df) / total_rows:.2%}")
print(f"Test : {len(test_df) / total_rows:.2%}")

In [7]:
# Affiche la proportion de chaque espèce dans le dataset complet.
print("Dataset complet")
display(
    dataframe["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans le train.
print("Train")
display(
    train_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans la validation.
print("Validation")
display(
    validation_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans le test.
print("Test")
display(
    test_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

In [8]:
# Convertit les DataFrames Pandas en tables PyArrow.
#
# preserve_index=False évite d'ajouter l'index Pandas
# comme nouvelle colonne dans le fichier Parquet.
train_table = pa.Table.from_pandas(
    train_df,
    preserve_index=False
)

validation_table = pa.Table.from_pandas(
    validation_df,
    preserve_index=False
)

test_table = pa.Table.from_pandas(
    test_df,
    preserve_index=False
)


In [9]:
print(train_df)


# Sauvegarde le dataset d'entraînement dans S3.
pq.write_table(
    train_table,
    TRAIN_PATH,
    filesystem=s3,
    compression="zstd"
)

# Sauvegarde le dataset de validation dans S3.
pq.write_table(
    validation_table,
    VALIDATION_PATH,
    filesystem=s3,
    compression="zstd"
)

# Sauvegarde le dataset de test dans S3.
pq.write_table(
    test_table,
    TEST_PATH,
    filesystem=s3,
    compression="zstd"
)


print("Les trois fichiers ont été sauvegardés dans S3.")